# 01 — Classificadores tradicionais (TF-IDF)

Grupo tradicional do projeto: **TF-IDF + Regressão Logística**, **TF-IDF + SVM linear** e **TF-IDF + Naive Bayes Multinomial**, aplicados ao Fake.br (texto normalizado por tamanho).

Estrutura inspirada no notebook do professor (`referencia/03d_ClassificacaoTextos-Spacy-scikit-learn.ipynb`: vários modelos avaliados com as mesmas métricas), com estas mudanças:

| Notebook do professor | Este notebook | Por quê |
|---|---|---|
| spaCy em inglês | `pt_core_news_sm` | O corpus é em português |
| TF-IDF ajustado no dataset inteiro | TF-IDF dentro de um `Pipeline` | Evita vazamento do teste para o treino |
| `train_test_split` sem semente | Divisão oficial (`data/split.csv`) | Mesma divisão do grupo de DL (ADR 0002) |
| Hiperparâmetros padrão | `GridSearchCV` + `GroupKFold` no treino | Ajuste justo, sem separar pares alinhados |
| Importância de features do RandomForest | Coeficientes da LogReg e do SVM | RF está fora do escopo |

**Protocolo:** o ajuste usa só o **treino** (validação cruzada com 5 partições agrupadas por par, otimizando F1). O **teste** só é usado na avaliação final. A **validação** não é usada aqui (ela serve para o early stopping do grupo de DL). Classe positiva = **notícia falsa** (`rotulo = 1`).

## 1. Setup

In [ ]:
# Setup: no Colab, clona o repositório do projeto e entra nele.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/fakebr-classificacao"):
        !git clone -q https://github.com/LeanderHeitor/fakebr-classificacao.git /content/fakebr-classificacao
    %cd /content/fakebr-classificacao
    !git pull -q

In [ ]:
# spaCy em português (no Colab o spaCy já vem instalado; só falta o modelo)
import importlib.util, subprocess, sys
if importlib.util.find_spec("pt_core_news_sm") is None:
    subprocess.run([sys.executable, "-m", "spacy", "download", "-q", "pt_core_news_sm"], check=True)

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import LinearSVC

from src.dados import SEMENTE, carregar_corpus

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 200)
DIR_RESULTADOS = Path("resultados")
DIR_RESULTADOS.mkdir(exist_ok=True)

## 2. Dados

In [ ]:
df = carregar_corpus()
treino = df[df.split == "treino"].reset_index(drop=True)
teste = df[df.split == "teste"].reset_index(drop=True)
print(f"treino: {len(treino)} notícias | teste: {len(teste)} notícias")
treino.head(3)

## 3. Pré-processamento

Duas **variantes de pré-processamento** (ver `GLOSSARY.md`), e a melhor é escolhida pela validação cruzada:

- **texto limpo:** minúsculas, sem pontuação e sem números
- **texto lematizado:** texto limpo, lematizado e sem stopwords

Os dois são calculados uma vez para o corpus inteiro. Isso não vaza informação, porque cada notícia é processada isoladamente.

In [ ]:
nlp = spacy.load("pt_core_news_sm", disable=["parser", "ner"])

def preprocessar(textos):
    limpos, lematizados = [], []
    for doc in nlp.pipe(textos, batch_size=64):
        tokens = [t for t in doc if not (t.is_punct or t.is_space or t.like_num or t.is_digit)]
        limpos.append(" ".join(t.lower_ for t in tokens))
        lematizados.append(" ".join(t.lemma_.lower() for t in tokens if not t.is_stop))
    return limpos, lematizados

for parte in (treino, teste):
    parte["texto_limpo"], parte["texto_lematizado"] = preprocessar(parte.texto.tolist())

treino[["texto", "texto_limpo", "texto_lematizado"]].head(2)

## 4. Ajuste dos modelos

Cada modelo é um `Pipeline` **seletor de variante → TF-IDF → classificador**. A grade combina:

- a variante de pré-processamento (limpo × lematizado)
- o TF-IDF: `ngram_range` (unigramas × uni+bigramas) e `min_df` (1 × 3, valor que o paper original achou neutro)
- o hiperparâmetro de regularização do classificador (`C` ou `alpha`)

A validação cruzada é `GroupKFold(5)` com grupo = **par alinhado**, então as duas notícias de um par nunca ficam separadas entre treino e validação.

In [ ]:
def selecionar(X, coluna):
    return X[coluna]

def montar_pipeline(classificador):
    return Pipeline([
        ("variante", FunctionTransformer(selecionar, kw_args={"coluna": "texto_limpo"})),
        ("tfidf", TfidfVectorizer()),
        ("clf", classificador),
    ])

GRADE_COMUM = {
    "variante__kw_args": [{"coluna": "texto_limpo"}, {"coluna": "texto_lematizado"}],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 3],
}

MODELOS = {
    "logreg": (LogisticRegression(max_iter=2000, random_state=SEMENTE), {"clf__C": [0.1, 1, 10, 100]}),
    "svm":    (LinearSVC(random_state=SEMENTE),                          {"clf__C": [0.01, 0.1, 1, 10]}),
    "nb":     (MultinomialNB(),                                          {"clf__alpha": [0.01, 0.1, 0.5, 1.0]}),
}

NOMES = {"logreg": "Regressão Logística", "svm": "SVM linear", "nb": "Naive Bayes"}
CV = GroupKFold(n_splits=5)

In [ ]:
%%time
buscas = {}
for nome, (clf, grade_clf) in MODELOS.items():
    busca = GridSearchCV(montar_pipeline(clf), {**GRADE_COMUM, **grade_clf},
                         scoring="f1", cv=CV, n_jobs=-1)
    busca.fit(treino, treino.rotulo, groups=treino.par)
    buscas[nome] = busca
    print(f"{NOMES[nome]:20s} F1 (CV) = {busca.best_score_:.4f}")

In [ ]:
def resumir_parametros(p):
    return {
        "variante": p["variante__kw_args"]["coluna"].replace("texto_", ""),
        "ngram_range": p["tfidf__ngram_range"],
        "min_df": p["tfidf__min_df"],
        "regularização": p.get("clf__C", p.get("clf__alpha")),
    }

linhas = []
for nome, b in buscas.items():
    i = b.best_index_
    linhas.append({"modelo": NOMES[nome],
                   "F1 CV (média)": b.cv_results_["mean_test_score"][i],
                   "F1 CV (desvio)": b.cv_results_["std_test_score"][i],
                   **resumir_parametros(b.best_params_)})
pd.DataFrame(linhas).set_index("modelo").round(4)

Comparação das variantes de pré-processamento: melhor F1 de validação cruzada de cada modelo com cada variante.

In [ ]:
variantes = pd.DataFrame({
    NOMES[nome]: pd.DataFrame(b.cv_results_)
        .assign(variante=lambda d: d.param_variante__kw_args.map(lambda k: k["coluna"].replace("texto_", "")))
        .groupby("variante").mean_test_score.max()
    for nome, b in buscas.items()
}).T
variantes.round(4)

## 5. Avaliação no conjunto de teste

Cada modelo é reajustado com os melhores hiperparâmetros em todo o treino (`refit` do GridSearch) e avaliado **uma única vez** no teste.

Para o arquivo de predições, o `score` precisa ser uma probabilidade de "falsa". A LogReg e o NB já têm `predict_proba`. O SVM não tem, então o `score` dele vem de um `CalibratedClassifierCV` ajustado no treino com as mesmas partições agrupadas. O `y_pred` do SVM continua sendo o do modelo ajustado.

In [ ]:
X_teste, y_teste = teste, teste.rotulo.values
predicoes, scores = {}, {}

for nome, b in buscas.items():
    modelo = b.best_estimator_
    predicoes[nome] = modelo.predict(X_teste)
    if nome == "svm":
        particoes = list(CV.split(treino, treino.rotulo, groups=treino.par))
        calibrado = CalibratedClassifierCV(modelo, cv=particoes).fit(treino, treino.rotulo)
        scores[nome] = calibrado.predict_proba(X_teste)[:, 1]
    else:
        scores[nome] = modelo.predict_proba(X_teste)[:, 1]

metricas = pd.DataFrame([{
    "modelo": NOMES[n],
    "acurácia": accuracy_score(y_teste, p),
    "precisão (falsa)": precision_score(y_teste, p),
    "revocação (falsa)": recall_score(y_teste, p),
    "F1 (falsa)": f1_score(y_teste, p),
    "F1 macro": f1_score(y_teste, p, average="macro"),
} for n, p in predicoes.items()]).set_index("modelo")
metricas.round(4)

> **Referência:** o paper original do Fake.br (Monteiro et al., 2018) obteve **0,88** de F1 e de acurácia com LinearSVC + bag of words booleano no texto normalizado (5-fold CV sem agrupamento por par). Ver `docs/adr/0001`.

In [ ]:
for nome, p in predicoes.items():
    print(f"=== {NOMES[nome]} ===")
    print(classification_report(y_teste, p, target_names=["verdadeira", "falsa"], digits=4))

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(15, 4))
for eixo, (nome, p) in zip(eixos, predicoes.items()):
    ConfusionMatrixDisplay.from_predictions(y_teste, p, display_labels=["verdadeira", "falsa"],
                                            cmap="Blues", colorbar=False, ax=eixo)
    eixo.set_title(NOMES[nome])
    eixo.set_xlabel("Predito"); eixo.set_ylabel("Real")
plt.tight_layout(); plt.show()

## 6. Arquivos de predição

Contrato combinado com o grupo de DL: `resultados/predictions_<modelo>.csv` com as colunas `id, y_true, y_pred, score` sobre o conjunto de teste, sendo `score` a probabilidade de "falsa". O notebook `04_comparacao` lê esses arquivos.

In [ ]:
for nome in predicoes:
    arquivo = DIR_RESULTADOS / f"predictions_{nome}.csv"
    pd.DataFrame({"id": teste.id, "y_true": y_teste,
                  "y_pred": predicoes[nome], "score": scores[nome].round(6)}).to_csv(arquivo, index=False)
    print("salvo:", arquivo)

**Para enviar os resultados ao repositório** (uma vez, depois de rodar tudo): no Colab, o jeito mais simples é baixar os 3 CSVs (painel de arquivos à esquerda → `fakebr-classificacao/resultados`) e subir pelo GitHub (*Add file → Upload files*, na pasta `resultados/`). Se rodar localmente, basta `git add resultados && git commit && git push`.

## 7. Palavras mais indicativas de cada classe

Coeficientes dos modelos lineares: valores positivos empurram para **falsa** e negativos para **verdadeira**. As palavras estão na forma da variante escolhida (lematizada ou não).

In [ ]:
def top_termos(busca, n=20):
    modelo = busca.best_estimator_
    termos = modelo.named_steps["tfidf"].get_feature_names_out()
    coef = modelo.named_steps["clf"].coef_.ravel()
    ordem = np.argsort(coef)
    return (pd.Series(coef[ordem[-n:]], termos[ordem[-n:]]),   # falsa
            pd.Series(coef[ordem[:n]], termos[ordem[:n]]))     # verdadeira

fig, eixos = plt.subplots(2, 2, figsize=(14, 12))
for linha, nome in enumerate(["logreg", "svm"]):
    falsa, verdadeira = top_termos(buscas[nome])
    falsa.plot.barh(ax=eixos[linha, 0], color="#c0392b")
    eixos[linha, 0].set_title(f"{NOMES[nome]} — indicam FALSA")
    verdadeira.iloc[::-1].plot.barh(ax=eixos[linha, 1], color="#2471a3")
    eixos[linha, 1].set_title(f"{NOMES[nome]} — indicam VERDADEIRA")
plt.tight_layout(); plt.show()

## 8. Análise de erros

Feita sobre o **melhor modelo segundo a validação cruzada**. A escolha não pode ser pelo teste, senão o teste deixa de ser uma avaliação independente.

In [ ]:
melhor = max(buscas, key=lambda n: buscas[n].best_score_)
print("Melhor modelo (F1 CV):", NOMES[melhor])

analise = teste.assign(y_pred=predicoes[melhor], score=scores[melhor])
analise["erro"] = analise.rotulo != analise.y_pred
analise["tipo_erro"] = np.select(
    [analise.erro & (analise.rotulo == 0), analise.erro & (analise.rotulo == 1)],
    ["falso positivo (verdadeira → falsa)", "falso negativo (falsa → verdadeira)"], "acerto")
analise.tipo_erro.value_counts()

### 8.1 Taxa de erro por categoria

Atenção: categorias com poucas notícias no teste (economia e religião têm 6 cada) não permitem conclusões. A coluna `n` mostra o tamanho de cada uma.

In [ ]:
(analise.groupby("categoria")
    .agg(n=("erro", "size"), erros=("erro", "sum"), taxa_erro=("erro", "mean"))
    .sort_values("taxa_erro", ascending=False)
    .round(3))

### 8.2 Leitura qualitativa

Os erros em que o modelo estava **mais confiante** (score mais distante de 0,5) são os mais informativos para a discussão.

In [ ]:
erros = analise[analise.erro].assign(confianca=lambda d: (d.score - 0.5).abs())
colunas = ["id", "categoria", "score", "texto"]
print("Verdadeiras classificadas como FALSAS:")
display(erros[erros.rotulo == 0].nlargest(5, "confianca")[colunas])
print("Falsas classificadas como VERDADEIRAS:")
display(erros[erros.rotulo == 1].nlargest(5, "confianca")[colunas])

### 8.3 Anotações para o relatório

*(preencher depois de ler os exemplos acima)*

- Padrões nos falsos positivos:
- Padrões nos falsos negativos:
- O que as palavras da seção 7 sugerem sobre o que o modelo aprendeu (conteúdo × estilo):